In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import seaborn as sns

In [2]:
# Load snow totals
snow = pd.read_parquet(r"data\snotel_daily.parquet")

In [3]:
# Load groundwater
gwl = pd.read_parquet(r"data\gwl_daily.parquet")

In [13]:
# Load climate
noaa_clean = pd.read_parquet(r"data\noaa_clean_daily.parquet")

In [8]:
# Load streamflow
flow = pd.read_parquet(r"data\streamflow_daily.parquet")

In [14]:
noaa_clean.columns

Index(['station', 'name', 'elevation', 'date', 'temp_max', 'temp_min',
       'precipitation', 'month_num'],
      dtype='object')

In [15]:
noaa_clean['name'].unique()

array(['DILLON 1 E, CO US', 'GRAND JUNCTION WALKER FIELD, CO US',
       'GRAND LAKE 6 SSW, CO US'], dtype=object)

In [16]:
noaa_clean = (
    noaa_clean
    .sort_values(['station', 'date'])
    .copy()
)

In [17]:
def add_daily_features(df):

    df = df.sort_values(['station', 'date']).copy()

    # Mean temperature
    df['temp_mean'] = (
        df['temp_max'] + df['temp_min']
    ) / 2

    # 1-day lags
    df['temp_mean_lag1'] = (
        df.groupby('station')['temp_mean']
        .shift(1)
    )

    df['precip_lag1'] = (
        df.groupby('station')['precipitation']
        .shift(1)
    )

    # Rolling temperature means
    df['temp_7d_mean'] = (
        df.groupby('station')['temp_mean']
        .transform(
            lambda x: x.rolling(7, min_periods=5).mean()
        )
    )

    df['temp_14d_mean'] = (
        df.groupby('station')['temp_mean']
        .transform(
            lambda x: x.rolling(14, min_periods=10).mean()
        )
    )

    # Rolling precipitation totals
    df['precip_7d'] = (
        df.groupby('station')['precipitation']
        .transform(
            lambda x: x.rolling(7, min_periods=5).sum()
        )
    )

    df['precip_14d'] = (
        df.groupby('station')['precipitation']
        .transform(
            lambda x: x.rolling(14, min_periods=10).sum()
        )
    )

    df['precip_30d'] = (
        df.groupby('station')['precipitation']
        .transform(
            lambda x: x.rolling(30, min_periods=20).sum()
        )
    )

    # Temperature / precipitation interaction
    df['temp_precip_interaction'] = (
        df['temp_7d_mean'] * df['precip_7d']
    )

    # Cyclical seasonality
    df['day_of_year'] = df['date'].dt.dayofyear

    df['doy_sin'] = np.sin(
        2 * np.pi * df['day_of_year'] / 365.25
    )

    df['doy_cos'] = np.cos(
        2 * np.pi * df['day_of_year'] / 365.25
    )

    return df

noaa_features = add_daily_features(noaa_clean)

In [18]:
noaa_features[
    [
        'date',
        'station',
        'temp_mean',
        'temp_7d_mean',
        'precip_7d',
        'precip_14d',
        'temp_mean_lag1',
        'precip_lag1'
    ]
].head(20)

,date,station,temp_mean,temp_7d_mean,precip_7d,precip_14d,temp_mean_lag1,precip_lag1
0,1990-01-01,USC00052281,11.5,NaN,NaN,NaN,NaN,NaN
1,1990-01-02,USC00052281,19.0,NaN,NaN,NaN,11.5,0.00
2,1990-01-03,USC00052281,17.5,NaN,NaN,NaN,19.0,0.00
3,1990-01-04,USC00052281,0.0,NaN,NaN,NaN,17.5,0.00
4,1990-01-05,USC00052281,0.5,9.700000,0.00,NaN,0.0,0.00
5,1990-01-06,USC00052281,4.5,8.833333,0.00,NaN,0.5,0.00
6,1990-01-07,USC00052281,11.0,9.142857,0.00,NaN,4.5,0.00
7,1990-01-08,USC00052281,16.5,9.857143,0.00,NaN,11.0,0.00
8,1990-01-09,USC00052281,25.5,10.785714,0.13,NaN,16.5,0.00
9,1990-01-10,USC00052281,29.0,12.428571,0.13,0.13,25.5,0.13


In [ ]:
feature_cols = [
    'temp_mean',
    'temp_mean_lag1',
    'temp_7d_mean',
    'temp_14d_mean',
    'precipitation',
    'precip_lag1',
    'precip_7d',
    'precip_14d',
    'precip_30d',
    'temp_precip_interaction',
    'doy_sin',
    'doy_cos'
]

These features are useful because they describe not just “today’s weather,” but also the recent weather history that controls snow accumulation and melt.
- temp_mean: the day’s average temperature. This matters because warmer conditions promote melt and can shift precipitation from snow toward rain, while colder conditions favor snow retention.
- temp_mean_lag1: yesterday’s mean temperature. Snowpack responds to sustained conditions, not just one isolated day, so yesterday’s temperature gives the model some short-term memory.
- temp_7d_mean: average temperature over the last 7 days. This captures short warm or cold spells. A week of unusually warm weather can reduce SWE even if today is cool.
- temp_14d_mean: average temperature over the last 14 days. This captures more persistent temperature patterns and helps distinguish a brief warm day from a prolonged melt period.
- precipitation: the current day’s precipitation. This can directly contribute to snow accumulation if temperatures are cold enough.
- precip_lag1: yesterday’s precipitation. This lets the model account for very recent storm activity that may still be affecting snowpack.
- precip_7d: total precipitation over the previous 7 days. This represents recent storm accumulation and is often more meaningful for snowpack than a single day of precipitation.
- precip_14d: total precipitation over the previous 14 days. This captures a longer accumulation window and helps represent sustained wet or dry periods.
- precip_30d: total precipitation over the previous 30 days. This gives the model a broader measure of antecedent moisture and storm history. For low-snow prediction, a dry month leading into the current date can be important.
- temp_precip_interaction: combines recent temperature and precipitation. This is useful because precipitation has very different implications depending on temperature. Wet and cold conditions may build snowpack, while wet and warm conditions may produce rain instead.
- doy_sin: sine transformation of day of year. This represents seasonality smoothly, so the model can recognize where you are in the annual snow cycle without treating December 31 and January 1 as far apart.
- doy_cos: cosine transformation of day of year. This works together with doy_sin to represent the circular annual cycle.

In [19]:
noaa_features['date'] = pd.to_datetime(noaa_features['date'])
snow['date'] = pd.to_datetime(snow['date'])

# To basin

In [20]:
basin_snow_daily = (
    snow
    .groupby('date')
    .agg(
        basin_swe=('swe', 'mean'),
        basin_precip_increment=('precip_increment', 'mean'),
        n_snow_stations=('station', 'nunique')
    )
    .reset_index()
)

In [21]:
climate_daily = (
    noaa_features
    .groupby('date')
    .agg(
        temp_mean=('temp_mean', 'mean'),
        temp_mean_lag1=('temp_mean_lag1', 'mean'),
        temp_7d_mean=('temp_7d_mean', 'mean'),
        temp_14d_mean=('temp_14d_mean', 'mean'),

        precipitation=('precipitation', 'mean'),
        precip_lag1=('precip_lag1', 'mean'),
        precip_7d=('precip_7d', 'mean'),
        precip_14d=('precip_14d', 'mean'),
        precip_30d=('precip_30d', 'mean'),

        temp_precip_interaction=('temp_precip_interaction', 'mean'),
        doy_sin=('doy_sin', 'mean'),
        doy_cos=('doy_cos', 'mean'),

        n_climate_stations=('station', 'nunique')
    )
    .reset_index()
)

In [26]:
basin_daily = basin_snow_daily.merge(
    climate_daily,
    on='date',
    how='inner'
)

# To station

In [23]:
station_daily = snow.merge(
    climate_daily,
    on='date',
    how='inner'
)

In [24]:
snow_by_station = {
    station: df.copy()
    for station, df in station_daily.groupby('station')
}

In [27]:
from pathlib import Path

out_dir = Path("data/model_dfs")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Station-level merged dataset
station_daily.to_parquet(
    out_dir / "station_daily_model.parquet",
    index=False
)

# 2. Basin-level merged dataset
basin_daily.to_parquet(
    out_dir / "basin_daily_model.parquet",
    index=False
)

# 3. Climate-only daily features
climate_daily.to_parquet(
    out_dir / "climate_daily_features.parquet",
    index=False
)

# 4. Basin snow daily
basin_snow_daily.to_parquet(
    out_dir / "basin_snow_daily.parquet",
    index=False
)

print("Saved model dataframes to:", out_dir.resolve())

Saved model dataframes to: C:\Users\katel\extreme climate\data\model_dfs


In [28]:
station_dir = out_dir / "stations"
station_dir.mkdir(parents=True, exist_ok=True)

for station, df in station_daily.groupby('station'):

    safe_name = (
        station
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
    )

    df.to_parquet(
        station_dir / f"{safe_name}_model.parquet",
        index=False
    )

    print("Saved:", safe_name)

Saved: Copper_Mountain
Saved: Fremont_Pass
Saved: Grizzly_Peak
Saved: Mesa_Lakes
